In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("training.csv")

In [5]:
df.head()

,Source Port,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Inbound,Label
0,68,67,17.0,134415183,40694755.0,5.0,0.0,1500.0,0.0,300.0,...,0.000000,0.000000,0.0,0.0,10173688.75,3.593932e+06,14671724.0,6369775.0,0.0,0.0
1,0,0,0.0,4145142372,113871580.0,52.0,0.0,0.0,0.0,0.0,...,6.083333,0.668558,7.0,5.0,9489291.75,2.133550e+05,9806952.0,9169328.0,0.0,0.0
2,0,0,0.0,3033854500,112936309.0,39.0,0.0,0.0,0.0,0.0,...,4.916667,0.514929,6.0,4.0,9411353.75,2.927876e+05,9869393.0,9048974.0,0.0,0.0
3,59751,443,6.0,3455527062,24944828.0,8.0,6.0,37.0,0.0,31.0,...,20876.000000,313.955411,21098.0,20654.0,10009953.00,3.139554e+02,10010175.0,10009731.0,0.0,0.0
4,50304,53,17.0,3995656861,20779.0,2.0,2.0,86.0,366.0,43.0,...,0.000000,0.000000,0.0,0.0,0.00,0.000000e+00,0.0,0.0,0.0,0.0


In [7]:
df.columns = df.columns.str.strip()

# Remove rows where Label is 1.0, 2.0, 3.0, or 5.0
df = df[~df["Label"].isin([1.0, 2.0, 3.0, 5.0])]

# Save to new CSV (optional)
df.to_csv("cleaned_file.csv", index=False)

In [12]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# --- Step 1: Load CSV file ---
df = pd.read_csv("updated_labels.csv")  # ← change filename if needed

# --- Step 2: Clean column names ---
df.columns = df.columns.str.strip()

# --- Step 3: Keep only numeric columns ---
df = df.select_dtypes(include=['number'])

# --- Step 4: Handle missing values ---
df = df.fillna(0)

# --- Step 5: Separate features and target ---
if "Label" not in df.columns:
    raise KeyError("⚠️ 'Label' column not found. Check your CSV column names!")

X = df.drop(columns=["Label"])
y = df["Label"]

# Convert labels to integers if they are floats (like 0.0, 1.0)
y = y.astype(int)

# --- Step 6: Split data into train/test ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Step 7: Train XGBoost model ---
model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)
model.fit(X_train, y_train)

# --- Step 8: Feature importance analysis ---
importance = model.feature_importances_
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

top_features = feature_importance.head(10)
print("\n📊 Top 10 Decisive Features:\n", top_features)

# --- Step 9: Re-train using only top 10 features ---
X_train_top = X_train[top_features["Feature"]]
X_test_top = X_test[top_features["Feature"]]

model_top = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)
model_top.fit(X_train_top, y_train)

# --- Step 10: Evaluate accuracy ---
y_pred = model_top.predict(X_test_top)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Accuracy using Top 10 Features: {accuracy:.4f}")
print("\nRemaining features used for detection:")
print(list(top_features["Feature"]))


C:\Users\amrut\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:59:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



📊 Top 10 Decisive Features:
                    Feature  Importance
69                 Inbound    0.513972
46          URG Flag Count    0.296480
0              Source Port    0.076653
45          ACK Flag Count    0.071457
57  Init_Win_bytes_forward    0.021297
1         Destination Port    0.003688
2                 Protocol    0.002963
43          SYN Flag Count    0.002882
23           Fwd IAT Total    0.002698
6   Total Backward Packets    0.002188


C:\Users\amrut\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:59:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



✅ Accuracy using Top 10 Features: 1.0000

Remaining features used for detection:
['Inbound', 'URG Flag Count', 'Source Port', 'ACK Flag Count', 'Init_Win_bytes_forward', 'Destination Port', 'Protocol', 'SYN Flag Count', 'Fwd IAT Total', 'Total Backward Packets']
